# 2 · LangGraph — Stateful Agents

In notebook 1 we saw LangChain make the model *request* a tool — but **we** had to run
it and feed the result back. LangGraph automates that loop, and much more.

**What you'll learn**
1. **State graph from scratch** — nodes, edges, and how state flows
2. **Conditional edges** — branching / routing
3. **Prebuilt ReAct agent** — the reason→act→observe loop, for free
4. **Memory** — a checkpointer so follow-ups work from context
5. **Human-in-the-loop** — pause for approval before a tool runs

> ▶️ Run **top to bottom** with **Shift + Enter**.

## 0 · Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from config import get_langchain_llm, assert_key
assert_key()
llm = get_langchain_llm()
print("Gateway ready.")

## 1 · A state graph from scratch

A LangGraph app is a **graph**: **nodes** are functions that read the shared **state**
and return updates to it; **edges** decide what runs next.

Here: `write → critique`. Watch how each node adds a field to the state.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    topic: str
    draft: str
    critique: str

def write(state: State) -> dict:
    draft = llm.invoke(f"Write a one-sentence product tagline about: {state['topic']}").content
    return {"draft": draft}          # <- node returns an UPDATE to the state

def critique(state: State) -> dict:
    c = llm.invoke(f"Critique this tagline in one short line: {state['draft']}").content
    return {"critique": c}

g = StateGraph(State)
g.add_node("write", write)
g.add_node("critique", critique)
g.add_edge(START, "write")           # entry
g.add_edge("write", "critique")      # write -> critique
g.add_edge("critique", END)          # done
app = g.compile()

result = app.invoke({"topic": "a warehouse security robot"})
print("DRAFT   :", result["draft"])
print("CRITIQUE:", result["critique"])

> **Key idea:** the **state** is the single source of truth that flows through the graph.
> Each node reads it and returns a partial update. (Open this graph in **LangGraph Studio**
> with `langgraph dev` to see it visually.)

## 2 · Conditional edges — branching

Real agents don't run a straight line — they **decide** what to do next. A
`add_conditional_edges` uses a **router function** to pick the next node from the state.

In [ ]:
from typing import Literal

class RouteState(TypedDict):
    question: str
    answer: str

def router(state: RouteState) -> Literal["math", "chat"]:
    # trivial, deterministic router (no LLM) so the branching is easy to see
    return "math" if any(ch.isdigit() for ch in state["question"]) else "chat"

def math_node(state): return {"answer": "→ routed to the MATH branch (a calculator agent)."}
def chat_node(state): return {"answer": "→ routed to the CHAT branch (a general agent)."}

g2 = StateGraph(RouteState)
g2.add_node("classify", lambda s: {})     # entry node
g2.add_node("math", math_node)
g2.add_node("chat", chat_node)
g2.add_edge(START, "classify")
g2.add_conditional_edges("classify", router, {"math": "math", "chat": "chat"})
g2.add_edge("math", END)
g2.add_edge("chat", END)
app2 = g2.compile()

print(app2.invoke({"question": "What is 12 * 4?"})["answer"])
print(app2.invoke({"question": "Tell me about robots"})["answer"])

## 3 · The prebuilt ReAct agent (the loop, automated)

`create_agent` builds the full **reason → act → observe → repeat** loop for you:
the model calls tools, LangGraph runs them, feeds results back, and loops until it has
a final answer. This is the manual tool loop from notebook 1 — done for you.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def inventory(sku: str) -> int:
    'Units in stock for a product SKU (scout/hauler/sentinel).'
    return {"scout": 12, "hauler": 3, "sentinel": 0}.get(sku.lower(), 0)

@tool
def price(sku: str) -> int:
    'List price in USD for a product SKU.'
    return {"scout": 18000, "hauler": 42000, "sentinel": 30000}.get(sku.lower(), 0)

agent = create_agent(llm, tools=[inventory, price])
out = agent.invoke({"messages": [("user", "How many Hauler units do we have, and what's the total value?")]})
print(out["messages"][-1].content)

## 4 · Memory — a checkpointer

Add a **checkpointer** and give each conversation a `thread_id`. Now the agent remembers
prior turns on that thread — so a follow-up with no subject still works.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

mem_agent = create_agent(llm, tools=[inventory, price], checkpointer=MemorySaver())
cfg = {"configurable": {"thread_id": "demo-1"}}

def ask(q):
    print("USER :", q)
    print("AGENT:", mem_agent.invoke({"messages": [("user", q)]}, cfg)["messages"][-1].content, "\n")

ask("How many Hauler units do we have?")
ask("And the Scout?")     # <- no subject; answered from memory

## 5 · Human-in-the-loop — approve before a tool runs

For risky actions you want a human gate. `interrupt_before=["tools"]` pauses the graph
**before** executing tools. You inspect what it *wants* to do, then resume (approve) by
invoking with `None`.

In [ ]:
hitl_agent = create_agent(
    llm, tools=[inventory, price],
    checkpointer=MemorySaver(),
    interrupt_before=["tools"],        # <- pause before the tool node
)
cfg2 = {"configurable": {"thread_id": "approval-1"}}

# 1) Run — it stops BEFORE running the tool
hitl_agent.invoke({"messages": [("user", "How many Sentinel units are in stock?")]}, cfg2)
snapshot = hitl_agent.get_state(cfg2)
print("Paused. Next step would be:", snapshot.next)
print("Tool the agent wants to call:", snapshot.values["messages"][-1].tool_calls)

# 2) A human approves -> resume by invoking with None
final = hitl_agent.invoke(None, cfg2)
print("\nAPPROVED ->", final["messages"][-1].content)

## 6 · Recap

- A LangGraph app = **nodes** (functions over shared **state**) + **edges** (what runs next).
- **Conditional edges** give branching; the model/graph decides the path.
- **`create_agent`** (LangChain's prebuilt ReAct agent) automates the tool loop you hand-wrote in notebook 1.
- **Checkpointer + thread_id** = memory across turns.
- **`interrupt_before`** = human approval — the "production" safety gate.

➡️ Next: run `langgraph dev` and open **LangGraph Studio** to *see* these graphs execute,
step through nodes, and fork/replay.

## 🧪 Your turn
1. **Extend §1:** add a `revise` node that rewrites the tagline using the critique, then wire `critique → revise → END`.
2. **Extend §3:** add a `discount(sku, qty)` tool and ask a multi-step question that needs it.
3. **Extend §5:** after the pause, instead of approving, print the state and **don't** resume — confirm the tool never ran (the approval gate works).

In [ ]:
# your code here